In [1]:
import control as ctrl
import numpy as np

# Análisis de los sistemas en su forma de espacio de estados

Supongamos que tenemos la siguiente función transferencia:

In [2]:
G=ctrl.tf([1,1],[1,2,3])
G

TransferFunction(
array([1, 1]),
array([1, 2, 3]),
outputs=1, inputs=1)

De los apuntes de teoría podemos escribir una representación en espacio de estados en la forma canónica de controlabilidad que tendrá el mismo comportamiento externo que $G$ de la siguiente manera:

In [3]:
Ac=np.array([[-2, -3],[1,0]])
Ac

array([[-2, -3],
       [ 1,  0]])

Antes de seguir, verificamos que los polos de $G$ sean los autovalores de $\mathbf {Ac}$

In [4]:
print(G.poles())
print(np.linalg.eigvals(Ac))

[-1.+1.41421356j -1.-1.41421356j]
[-1.+1.41421356j -1.-1.41421356j]


Ahora definimos el resto de las matrices

In [5]:
Bc=np.array([[1],[0]])
Bc

array([[1],
       [0]])

In [6]:
Cc=np.array([[1, 1]])
Cc

array([[1, 1]])

In [7]:
Dc=0

Vamos ahora a definir un sistema `sys_c` que se comporte igual que $G$:

In [8]:
sys_c=ctrl.ss(Ac,Bc,Cc,Dc)

Si `sys_c` tiene los mismos polos, los mismos ceros y la misma ganancia en estado estacionario, entonces se comportará igual que $G$.

Veamos los polos:

In [9]:
sys_c.poles()

array([-1.+1.41421356j, -1.-1.41421356j])

Ahora los ceros:

In [10]:
sys_c.zeros()

array([-1.+0.j])

Y la ganancia en estado estacionario en particular (o en cualquier frecuencia en general):

In [11]:
sys_c.dcgain()

np.float64(0.3333333333333333)

In [12]:
G.dcgain()

np.float64(0.3333333333333333)

Por lo tanto podemos ver que la función transferencia de `sys_c` será la misma que la de $G$. Es decir, `sys_c` se comporta **externamente** igual que $G$. Verificamos:

In [13]:
ctrl.tf(sys_c)

TransferFunction(
array([1., 1.]),
array([1., 2., 3.]),
outputs=1, inputs=1)

Vamos ahora a definir el sistema en espacio de estados que se comporte como $G$ en la forma canónica de observabilidad. De la teoría sabemos que:

In [14]:
Ao=Ac.T
Bo=Cc.T
Co=Bc.T
Do=Dc

In [15]:
sys_o=ctrl.ss(Ao,Bo,Co,Do)

In [16]:
sys_o.poles()

array([-1.+1.41421356j, -1.-1.41421356j])

In [17]:
sys_o.zeros()

array([-1.+0.j])

In [18]:
sys_o.dcgain()

np.float64(0.3333333333333333)

In [19]:
ctrl.tf(sys_o)

TransferFunction(
array([1., 1.]),
array([1., 2., 3.]),
outputs=1, inputs=1)

El módulo de control de Python puede transformar una función transferencia a una forma de estados (no necesariamente en alguna forma canónica). Esto se hace:

In [20]:
sys=ctrl.ss(G)
sys

StateSpace(
array([[-2.0000000e+00,  3.0000000e+00],
       [-1.0000000e+00, -1.7047851e-16]]),
array([[1.],
       [0.]]),
array([[ 1., -1.]]),
array([[0.]]),
states=2, outputs=1, inputs=1)

Además Python puede transformar un sistema en espacio de estados a alguna de las formas canónicas.

Por ejemplo, para escribir `sys` (que está ahora en espacio de estados) en su forma canónica de controlabilidad podemos hacer:

In [21]:
sys_c, Tc=ctrl.canonical_form(sys, 'reachable') # controlable
sys_c, Tc

(StateSpace(
 array([[-2., -3.],
        [ 1.,  0.]]),
 array([[1.],
        [0.]]),
 array([[1., 1.]]),
 array([[0.]]),
 states=2, outputs=1, inputs=1),
 array([[ 1.00000000e+00,  4.44089210e-16],
        [ 5.55111512e-17, -1.00000000e+00]]))

Notar que esta función necesita como primer argumento un sistema en espacio de estados.

Si queremos la forma canónica de observabilidad podemos hacer:

In [22]:
sys_o,To = ctrl.canonical_form(sys, 'observable')
sys_o, To

(StateSpace(
 array([[-2.,  1.],
        [-3.,  0.]]),
 array([[1.],
        [1.]]),
 array([[1., 0.]]),
 array([[0.]]),
 states=2, outputs=1, inputs=1),
 array([[ 1., -1.],
        [ 1.,  1.]]))

Existe otra forma canónica de representación en espacio de estados que se la conoce como forma canónica modal. Esta la podemos obtener haciendo:

In [23]:
sys_m, Tm = ctrl.canonical_form(sys, 'modal')
sys_m, Tm

(StateSpace(
 array([[-1.        ,  3.41421356],
        [-0.58578644, -1.        ]]),
 array([[ 0.92387953],
        [-0.38268343]]),
 array([[ 0.5411961 , -1.30656296]]),
 array([[0.]]),
 states=2, outputs=1, inputs=1),
 array([[ 0.92387953, -0.38268343],
        [ 0.38268343,  0.92387953]]))

Esta forma canónica lo que hace es aislar los modos unos de otros. Es decir, cada modo aparece en la diagonal. Sin embargo, como vemos en este caso, no tenemos una forma diagonal de la matriz $\mathbf A$. Esto sucede porque hay casos en que no se puede o no se desea aislarlos:

- en caso de que se tengan autovalores complejos conjugados, no se aíslan para que las matrices $\mathbf A$, $\mathbf B$ y $\mathbf C$ tengan coeficientes reales.
- en caso de que sean múltiples, en general no se pueden aislar porque no hay suficientes autovectores independientes para armar la transformación. Para estos casos se utiliza la forma de Jordan.

Vamos a ver otro ejemplo:

In [24]:
s=ctrl.tf('s')
G2 = 1/((s+1)*(s+2)*(s+3))
G2

TransferFunction(
array([1]),
array([ 1,  6, 11,  6]),
outputs=1, inputs=1)

Vemos que esta función transferencia tiene los polos en -1, -2 y -3. Transformemos a espacio de estados usando la función `ss`.

In [25]:
sys2 = ctrl.ss(G2)
sys2

StateSpace(
array([[ -6. ,   1.1,  -0.6],
       [-10. ,   0. ,   0. ],
       [  0. ,  -1. ,   0. ]]),
array([[-1.],
       [ 0.],
       [ 0.]]),
array([[ 0. ,  0. , -0.1]]),
array([[0.]]),
states=3, outputs=1, inputs=1)

Veamos al sistema en su forma canónica de controlabilidad.

In [26]:
ctrl.canonical_form(sys2, 'reachable')

(StateSpace(
 array([[ -6., -11.,  -6.],
        [  1.,   0.,   0.],
        [  0.,   1.,   0.]]),
 array([[1.],
        [0.],
        [0.]]),
 array([[-1.33226763e-17, -6.81273220e-17,  1.00000000e+00]]),
 array([[0.]]),
 states=3, outputs=1, inputs=1),
 array([[-1.00000000e+00,  6.88338275e-16,  3.12666559e-15],
        [ 3.55271368e-17,  1.00000000e-01, -6.95554725e-16],
        [-1.33226763e-17,  6.81273220e-18, -1.00000000e-01]]))

Ahora lo podemos ver en la forma canónica de observabilidad

In [27]:
ctrl.canonical_form(sys2, 'observable')

(StateSpace(
 array([[ -6.,   1.,   0.],
        [-11.,   0.,   1.],
        [ -6.,   0.,   0.]]),
 array([[0.],
        [0.],
        [1.]]),
 array([[1., 0., 0.]]),
 array([[0.]]),
 states=3, outputs=1, inputs=1),
 array([[ 0. ,  0. , -0.1],
        [-0. ,  0.1, -0.6],
        [-1. ,  0.6, -1.1]]))

Finalmente obtengamos la forma canónica modal:

In [28]:
sys2_m, Tm =ctrl.canonical_form(sys2, 'modal')
sys2_m.A

array([[-3.00000000e+00,  1.77635684e-15,  3.55271368e-15],
       [-7.10542736e-15, -2.00000000e+00, -3.55271368e-15],
       [-3.77475828e-15, -1.99840144e-15, -1.00000000e+00]])

```{note}
- La función `canonical_form` devuelve un sistema en la forma canónica solicitada y la matriz de transformación necesaria para obtenerlo
- en la diagonal de la matriz  $\mathbf A$ de la forma canónica modal tenemos los modos del sistema (polos de $G$ o autovalores de $A$)
- fuera de la diagonal $\mathbf A$ tenemos ceros o valores muy cercanos a ceros (debido a problemas numéricos en el cálculo).
- En caso de tener autovalores repetidos, tendremos una matriz $\mathbf{A}$ diagonal por bloques (forma de Jordan).
- En general, los autovalores complejos conjugados tampoco se ponen en la diagonal, para evitar tener matrices con valores complejos.
```

Verificamos el resto de las matrices:

In [29]:
sys2_m.B

array([[ 16.43928222],
       [-22.71563338],
       [ -7.08872344]])

In [30]:
sys2_m.C

array([[ 0.03041495,  0.04402255, -0.07053456]])

In [31]:
sys2_m.D

array([[0.]])

En esta forma canónica podemos interpretar la matriz $\mathbf B$ como la conexión de la entrada con cada modo del sistema y, de forma análoga, la matriz $\mathbf C$ como la conexión de cada modo con la salida del sistema.

## Ejemplo: controlabilidad y observabilidad en la conexión de sistemas

Supongamos que se tienen las siguientes funciones transferencia $G_1$ y $G_2$:

In [32]:
G1=ctrl.tf(1,[1,1])
G2=ctrl.tf([1,1],[1,2])
G1

TransferFunction(
array([1]),
array([1, 1]),
outputs=1, inputs=1)

In [33]:
G2

TransferFunction(
array([1, 1]),
array([1, 2]),
outputs=1, inputs=1)

Podemos ver que $G_1$ tiene un polo en -1 y $G_2$ tiene un polo en -2 y un cero en -1.

Ahora supongamos que queremos un nuevo sistema que:

- conecta la salida de $G_1$ con la entrada de $G_2$,
- su salida es la salida de $G_2$,
- su entrada es la entrada de $G_1$.

Para eso vamos a usar la función `interconnect`. Esta conecta sistemas en espacio de estados a partir de los nombres de sus señales: las entradas y salidas que tienen el mismo nombre se conectan entre sí. Entonces pasamos $G_1$ y $G_2$ a espacio de estados, nombrando sus señales de modo que la salida de $G_1$ (`v`) sea la entrada de $G_2$:

In [34]:
sys1 = ctrl.ss(G1, inputs='u', outputs='v', name='G1')
sys2 = ctrl.ss(G2, inputs='v', outputs='y', name='G2')

Y luego realizamos las conexiones, indicando cuáles son la entrada y la salida del sistema completo:

In [35]:
sys_c = ctrl.interconnect([sys1, sys2], inplist='u', outlist='y')
sys_c

<LinearICSystem sys[25]: ['u[0]'] -> ['y[0]']>

Analizamos los polos y ceros del sistema ya interconectado:

In [36]:
sys_c.poles()

array([-2.+0.j, -1.+0.j])

In [37]:
sys_c.zeros()

array([-1.+0.j])

Vemos que el sistema conserva los mismos polos y los mismos ceros que $G_1$ y $G_2$, lo cual era de esperarse, ya que su función transferencia es $G_1G_2$.

Obtengamos la forma canónica de controlabilidad del sistema:

In [38]:
sys_c, Tc=ctrl.canonical_form(sys_c, 'reachable')
sys_c

StateSpace(
array([[-3., -2.],
       [ 1.,  0.]]),
array([[1.],
       [0.]]),
array([[1., 1.]]),
array([[0.]]),
states=['G1_x[0]', 'G2_x[0]'], outputs=1, inputs=1)

Podemos ver que las matrices $ \mathbf A$, $\mathbf B$, $\mathbf C$ y $\mathbf D$ coinciden con lo visto en teoría.

Ahora obtengamos la forma canónica de observabilidad:

In [39]:
ctrl.canonical_form(sys_c, 'observable')

ValueError: Transformation matrix singular to working precision.

Podemos ver que este sistema no puede ser escrito en su forma canónica de observabilidad. Esto sucede porque el sistema no es observable.

```{admonition} Interpretación
:class: hint

Un sistema observable necesita tener evidencia de lo que sucede con cada modo en al menos una salida del sistema. Como el modo en -1 queda completamente tapado por el cero en esa misma frecuencia, este sistema no es observable: no hay ninguna evidencia del modo en -1 a la salida.

```

Ahora hagamos la conexión al revés (primero $G_2$ y luego $G_1$):

Para esto cambiamos los nombres de las señales: ahora la salida de $G_2$ (`v`) es la entrada de $G_1$:

In [40]:
sys1_o = ctrl.ss(G1, inputs='v', outputs='y', name='G1')
sys2_o = ctrl.ss(G2, inputs='u', outputs='v', name='G2')
sys_o = ctrl.interconnect([sys1_o, sys2_o], inplist='u', outlist='y')
sys_o

<LinearICSystem sys[31]: ['u[0]'] -> ['y[0]']>

Obtengamos la forma canónica observable:

In [41]:
sys_o, To=ctrl.canonical_form(sys_o, 'observable')
sys_o

StateSpace(
array([[-3.,  1.],
       [-2.,  0.]]),
array([[1.],
       [1.]]),
array([[1., 0.]]),
array([[0.]]),
states=['G1_x[0]', 'G2_x[0]'], outputs=1, inputs=1)

Y ahora la forma canónica controlable:

In [42]:
ctrl.canonical_form(sys_o, 'reachable')

ValueError: System not controllable to working precision.

Podemos ver que ahora no podemos obtener la forma canónica de controlabilidad.

```{admonition} Interpretación
:class: hint

La razón de esto es que no se puede excitar el modo -1, que ahora quedó tapado desde el lado de la entrada. Toda excitación que tenga "frecuencia generalizada -1" será bloqueada por el cero en -1. Por lo tanto este sistema no es controlable, ya que no se puede excitar el modo -1 desde esa entrada.
```

Es importante destacar que ambos sistemas presentan la misma función transferencia $G_1G_2$, pero uno es solo controlable y el otro es solo observable. Esto se debe a que la controlabilidad y la observabilidad son propiedades internas de un sistema.

```{attention}
La **controlabilidad** y la **observabilidad**  no pueden ser decididas a partir de modelos externos de un sistema como son las funciones transferencias.
```

Estas dos mismas propiedades pueden ser analizadas con la forma modal. Tomemos el sistema `sys_c`, donde la entrada está conectada a $G_1$ y la salida a $G_2$:

In [43]:
sys_mc, Tmc = ctrl.canonical_form(sys_c, 'modal')
sys_mc

StateSpace(
array([[-2.,  0.],
       [ 0., -1.]]),
array([[-2.23606798],
       [-1.41421356]]),
array([[-0.4472136,  0.       ]]),
array([[0.]]),
states=['G1_x[0]', 'G2_x[0]'], outputs=1, inputs=1)

Podemos ver que la matriz $\mathbf B$ tiene todos los valores distintos de 0. Esto significa que todos los modos están conectados a la entrada, por lo que deducimos que el sistema es controlable.

Pero podemos ver que $\mathbf C$ tiene un 0 (o un valor numéricamente despreciable) en la posición del modo -1 de la diagonal de $\mathbf A$. Esto quiere decir que no hay ninguna evidencia del modo -1 en la salida, por lo que el sistema no es observable.

Probemos ahora con la segunda conexión:

In [44]:
sys_mo, Tmo = ctrl.canonical_form(sys_o, 'modal')
sys_mo

StateSpace(
array([[-2.,  0.],
       [ 0., -1.]]),
array([[-1.41421356],
       [-0.        ]]),
array([[-0.70710678, -0.4472136 ]]),
array([[0.]]),
states=['G1_x[0]', 'G2_x[0]'], outputs=1, inputs=1)

Podemos ver ahora que la matriz $\mathbf C$ no tiene ningún valor igual a 0, por lo que ahora el sistema es observable (todos los modos están presentes de alguna manera en la salida). Pero tenemos un 0 (o un valor numéricamente despreciable) en la fila de $\mathbf B$ que se corresponde con el modo en -1. Esto quiere decir que no podemos excitar este modo. Por lo tanto no es controlable.

```{important}
La controlabilidad y la observabilidad pueden ser evaluadas con el sistema en su forma **canónica modal**, analizando las relaciones de las matrices $\mathbf{B}$ y $\mathbf{C}$ con la matriz $\mathbf{A}$.
```